# ML-04 — Search Intelligence Data Contract

**Lane: CTR / Engagement Opportunity Scoring (CTR focus).** This continues ML-02/03: help an editor prioritize visible pages for snippet/intent review. Here I verify a real warehouse slice and perform a small future-window leakage lesson. A low-CTR outcome is a measurement proxy, not an independently reviewed actionable edit.

Source: [FlyRank warehouse v20260703](https://huggingface.co/datasets/FlyRank/internship-warehouse), pinned to repository revision `50cbf7c3909d07be4d1b5906b4d09e882e5acbf2`. Only `month=2026-03` and `dim_clients` are read; June and the final-month `_sample` remain sealed. Public outputs contain aggregates and a five-row feature projection, never raw daily rows or identifying fields.

## 1. Unit of analysis + time window — five plain-words answers

1. **One row means:** the source row is one report date × pseudonymized client × content item. My feature row is one client × page at one historical decision moment, built by aggregating its early-March observations. IDs group rows; they are not model inputs.
2. **Tables:** `fact_content_daily_performance/month=2026-03/data_0.parquet` supplies search counts, positions and the analytics-availability flag. `dim_clients` supplies history-start context for the coverage audit. I do not join `dim_content` because its final snapshot metadata need not describe a page as it was in March.
3. **Windows:** features use **March 1–15, 2026**, inclusive. Assume a decision at the start of **March 19**, leaving March 16–18 unused for a conservative three-day reporting lag. Labels use **March 19–31**, inclusive, and become measurable in early April. This lag is a modeling assumption based on the release freshness policy, not measured historical ingestion timestamps.
4. **Predict/rank:** the quick classifier estimates a **later-window CTR < 0.5%** proxy for early-window pages with at least **500 impressions** and impression-weighted position **1–20**. Evaluation additionally requires at least **100 later-window impressions**. These cutoffs are declared policy choices, not optimized results. A future queue could order pages for human review; actual editorial precision@20 is still unavailable without independent judgments.
5. **Deliberately exclude:** all later-window measurements and the label from honest features. Also exclude IDs from learning, the fixed 90-day query table (window overlaps), product decisions, and June. For this small paired-measurement cohort, require `ga4_data_available IS TRUE` in the early window, even though the five inputs are search-only. This conservative restriction makes the availability check concrete but narrows generalization; it is not necessary for every CTR project.

Require all **15 early-window daily rows** to pass availability and count validity, and at least one observed positive position. Missing dates are not filled with zeros. Outcome eligibility is an offline evaluation restriction, never used to decide which pages enter the historical feature frame. Later coverage losses are reported separately.

In [1]:
# Dependencies: duckdb, huggingface_hub, pandas, numpy, scikit-learn, ipykernel.
# Install locally or in Colab before Run all; never put a credential in a cell.
import os
# Regular HTTP avoids the stalled Xet transfer seen on this machine.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
import hashlib
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display, Markdown
from huggingface_hub import get_token, snapshot_download

REPO_ID = 'FlyRank/internship-warehouse'
REVISION = '50cbf7c3909d07be4d1b5906b4d09e882e5acbf2'
MONTH = '2026-03'
FEATURE_START = pd.Timestamp('2026-03-01')
FEATURE_END = pd.Timestamp('2026-03-15')
DECISION_DATE = pd.Timestamp('2026-03-19')
OUTCOME_END = pd.Timestamp('2026-03-31')
FEATURE_COLS = ['log_impressions', 'ctr_pct', 'weighted_position',
                'position_std', 'active_day_fraction']
assert FEATURE_END < DECISION_DATE <= OUTCOME_END

# Optional directory containing the same approved release files; otherwise use HF cache.
local_root = os.environ.get('FLYRANK_WAREHOUSE_DIR')
if local_root:
    warehouse = Path(local_root).expanduser()
else:
    token = get_token()  # HF_TOKEN or `hf auth login`, kept outside the public notebook.
    try:
        from google.colab import userdata
        token = token or userdata.get('HF_TOKEN')
    except Exception:
        pass
    if not token:
        raise RuntimeError('Save a Hugging Face Read token with hf auth login or Colab Secret HF_TOKEN.')
    try:
        warehouse = Path(snapshot_download(
            repo_id=REPO_ID, repo_type='dataset', revision=REVISION, token=token,
            allow_patterns=['dim_clients.parquet',
                            'fact_content_daily_performance/month=2026-03/*.parquet'],
            max_workers=1))
    except Exception:
        raise RuntimeError('Warehouse access/download failed. Check gate approval and Read-token permissions.') from None
    del token

daily_files = sorted(warehouse.glob('fact_content_daily_performance/month=2026-03/*.parquet'))
client_file = warehouse / 'dim_clients.parquet'
assert len(daily_files) == 1 and daily_files[0].name == 'data_0.parquet'
assert client_file.exists()
con = duckdb.connect()
con.execute('SET threads = 4')
con.from_parquet([str(p) for p in daily_files], hive_partitioning=False).create_view('march_source')
con.from_parquet(str(client_file)).create_view('client_history')
needed = {'report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions',
          'gsc_clicks', 'gsc_avg_position', 'ga4_data_available'}
assert needed.issubset(set(con.table('march_source').columns))
file_provenance = pd.DataFrame([
    {'source': 'March daily partition' if p in daily_files else 'client dimension',
     'bytes': p.stat().st_size,
     'sha256': hashlib.file_digest(p.open('rb'), 'sha256').hexdigest()}
    for p in [*daily_files, client_file]
])
display(file_provenance)
print('Pinned March release loaded. No June/sample data read; no dataset written to Git.')
display(pd.DataFrame({"package": ["duckdb", "huggingface_hub", "pandas", "numpy", "scikit-learn"],
                      "executed_version": [version(p) for p in ["duckdb", "huggingface_hub", "pandas", "numpy", "scikit-learn"]]}))

/tmp/flyrank-ml02-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 1382.21it/s]

,source,bytes,sha256
0,March daily partition,124233710,6ebd367f05932238df05b193f9cfb9cc7839d7856b65ef...
1,client dimension,3377,b64a9f13c102b7563f1f632c20a8680cb4a9b2fa4def0f...


Pinned March release loaded. No June/sample data read; no dataset written to Git.


,package,executed_version
0,duckdb,1.5.6
1,huggingface_hub,2.1.1
2,pandas,3.0.6
3,numpy,2.4.6
4,scikit-learn,1.9.1


## 2. Fields: feature / label / context / excluded

| Bucket | Fields touched | Role / timing |
|---|---|---|
| Feature inputs | early-window `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | These feed exactly five aggregations below; counts are not averages of daily rates. |
| Features | `log_impressions`, `ctr_pct`, `weighted_position`, `position_std`, `active_day_fraction` | Only March 1–15 observations; five numeric inputs. |
| Label/proxy | later-window impressions, clicks, `future_ctr_pct`, `future_low_ctr` | Measured March 19–31. `future_low_ctr = future_ctr_pct < 0.5`, only where exposure/coverage are sufficient. Missing outcome is never a negative label. |
| Context | `report_date`, `client_hash_id`, `content_hash_id`, `ga4_data_available`, `gsc_data_start`, `ga4_data_start`, window row counts | Date/window selection, keys, split groups, availability and completeness checks; no learning from IDs. |
| Excluded | March 16–18 observations; June/sample; query-90d table; final-snapshot metadata; raw URLs/queries/client names; product scores; deliberate `label_echo` | Gap prevents timing overlap; future snapshots leak; raw fields are unnecessary/private; rule outputs are circular; `label_echo` is used only to demonstrate and then delete leakage. |

**Missing values:** NULL/false availability rows are excluded with `IS TRUE`. NULL/negative search counts are invalid, not zero. Position ≤0 or NULL is absent; it is not rank zero. Weighted position and volatility use only positive-impression days with positive position. A page with no usable positions stays missing and is excluded. Report the missing position-day rate separately by availability state in verification 3; show feature missingness after aggregation. No imputation is learned across train/test.

**Output:** a checked five-feature page frame, an honest held-out-client proxy score, and a visibly invalid score produced by one leaked column, with that column removed.

## 3. Exactly three verification queries, then features + trap

The following are the **three verification queries**. Their aggregate outputs are displayed. Setup creates views; the later SQL builds the feature frame and is not a fourth verification check.

### Verification 1 — does the daily grain hold?
Group every March row by the declared keys. Zero duplicate groups and zero NULL-key groups support the grain; an aggregate report avoids displaying IDs.

In [2]:
grain_sql = """
WITH grain AS (
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM march_source
    GROUP BY report_date, client_hash_id, content_hash_id
)
SELECT COUNT(*) AS distinct_daily_keys,
       COUNT(*) FILTER (WHERE n > 1) AS duplicate_key_groups,
       COALESCE(SUM(n - 1) FILTER (WHERE n > 1), 0) AS excess_rows,
       COUNT(*) FILTER (WHERE report_date IS NULL OR client_hash_id IS NULL
                              OR content_hash_id IS NULL) AS null_key_groups
FROM grain
"""
grain_check = con.sql(grain_sql).df()
display(grain_check)
assert grain_check.loc[0, 'duplicate_key_groups'] == 0
assert grain_check.loc[0, 'null_key_groups'] == 0
print('Observed: March source keys are unique and non-NULL.')

,distinct_daily_keys,duplicate_key_groups,excess_rows,null_key_groups
0,9841378,0,0.0,0


Observed: March source keys are unique and non-NULL.


### Verification 2 — row count, date span, and history support
Count the entire March source slice and both disjoint windows, plus how many March clients began GSC/GA4 after the proposed feature start (or have missing history metadata). Client-history data are collapsed by key before joining to prevent count inflation; conflicting duplicate history keys are counted.

In [3]:
counts_sql = """
WITH history AS (
    SELECT client_hash_id, MIN(gsc_data_start) AS gsc_data_start,
           MIN(ga4_data_start) AS ga4_data_start, COUNT(*) AS dimension_rows
    FROM client_history GROUP BY client_hash_id
)
SELECT COUNT(*) AS source_rows, MIN(f.report_date) AS first_date,
       MAX(f.report_date) AS last_date,
       COUNT(DISTINCT f.client_hash_id) AS clients,
       COUNT(DISTINCT (f.client_hash_id, f.content_hash_id)) AS page_keys,
       COUNT(*) FILTER (WHERE f.report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-15') AS feature_window_rows,
       COUNT(*) FILTER (WHERE f.report_date BETWEEN DATE '2026-03-19'
                          AND DATE '2026-03-31') AS outcome_window_rows,
       COUNT(DISTINCT f.client_hash_id) FILTER
           (WHERE h.gsc_data_start > DATE '2026-03-01') AS late_gsc_start_clients,
       COUNT(DISTINCT f.client_hash_id) FILTER
           (WHERE h.ga4_data_start > DATE '2026-03-01') AS late_ga4_start_clients,
       COUNT(DISTINCT f.client_hash_id) FILTER
           (WHERE h.gsc_data_start IS NULL) AS unknown_gsc_start_clients,
       COUNT(DISTINCT f.client_hash_id) FILTER
           (WHERE h.ga4_data_start IS NULL) AS unknown_ga4_start_clients,
       COUNT(DISTINCT f.client_hash_id) FILTER
           (WHERE h.dimension_rows > 1) AS duplicate_dimension_clients
FROM march_source f LEFT JOIN history h USING (client_hash_id)
"""
counts_check = con.sql(counts_sql).df()
display(counts_check.T.rename(columns={0: 'observed'}))
assert counts_check.loc[0, 'source_rows'] == grain_check.loc[0, 'distinct_daily_keys']
assert pd.Timestamp(counts_check.loc[0, 'first_date']) == FEATURE_START
assert pd.Timestamp(counts_check.loc[0, 'last_date']) == OUTCOME_END
assert counts_check.loc[0, 'duplicate_dimension_clients'] == 0
print(f"Observed March slice: {counts_check.loc[0, 'source_rows']:,} rows across "
      f"{counts_check.loc[0, 'clients']} clients; exact span shown above.")

,observed
source_rows,9841378
first_date,2026-03-01 00:00:00
last_date,2026-03-31 00:00:00
clients,55
page_keys,331437
feature_window_rows,4642255
outcome_window_rows,4238469
late_gsc_start_clients,5
late_ga4_start_clients,16
unknown_gsc_start_clients,2


Observed March slice: 9,841,378 rows across 55 clients; exact span shown above.


### Verification 3 — availability with `IS TRUE`, and patterned missingness
Display total and surviving counts for March, the feature window, and the outcome window. Also compare unavailable and available rows: missing/invalid search measurements and unusable position-days must not be silently interpreted as measured zeros. Requiring analytics tracking intentionally excludes GSC-only rows here.

In [4]:
availability_sql = """
WITH windows AS (
    SELECT 'whole March' AS period, * FROM march_source
    UNION ALL
    SELECT 'feature: Mar 01-15', * FROM march_source
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    UNION ALL
    SELECT 'outcome: Mar 19-31', * FROM march_source
    WHERE report_date BETWEEN DATE '2026-03-19' AND DATE '2026-03-31'
)
SELECT period,
       CASE WHEN ga4_data_available IS TRUE THEN 'available'
            ELSE 'false or NULL' END AS availability,
       COUNT(*) AS rows_before_filter_in_group,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_surviving_is_true,
       COUNT(DISTINCT client_hash_id) AS clients_in_group,
       COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS null_flag_rows,
       ROUND(100.0 * AVG(CASE WHEN gsc_impressions IS NULL OR gsc_clicks IS NULL
                                  OR gsc_impressions < 0 OR gsc_clicks < 0
                             THEN 1 ELSE 0 END), 4) AS invalid_count_pct,
       ROUND(100.0 * AVG(CASE WHEN gsc_impressions > 0 AND
                              (gsc_avg_position IS NULL OR gsc_avg_position <= 0)
                             THEN 1 ELSE 0 END), 4) AS missing_position_on_visible_row_pct
FROM windows
GROUP BY period, availability ORDER BY period, availability
"""
availability_check = con.sql(availability_sql).df()
display(availability_check)
march_availability = availability_check.loc[availability_check['period'].eq('whole March')]
survived = int(march_availability['rows_surviving_is_true'].sum())
before = int(march_availability['rows_before_filter_in_group'].sum())
assert before == counts_check.loc[0, 'source_rows']
print(f'Observed availability: {survived:,}/{before:,} March rows survive IS TRUE '
      f'({survived / before:.2%}). Unavailable is not zero engagement.')

,period,availability,rows_before_filter_in_group,rows_surviving_is_true,clients_in_group,null_flag_rows,invalid_count_pct,missing_position_on_visible_row_pct
0,feature: Mar 01-15,available,159060,159060,37,0,0.0,0.7871
1,feature: Mar 01-15,false or NULL,4483195,0,52,1761384,0.0,1.5607
2,outcome: Mar 19-31,available,198842,198842,40,0,0.0,0.7700
3,outcome: Mar 19-31,false or NULL,4039627,0,55,964213,0.0,1.8822
4,whole March,available,413966,413966,41,0,0.0,0.7986
5,whole March,false or NULL,9427412,0,55,3018741,0.0,1.6959


Observed availability: 413,966/9,841,378 March rows survive IS TRUE (4.21%). Unavailable is not zero engagement.


### Five features — available when?

| Feature | Definition | Available when? |
|---|---|---|
| `log_impressions` | log(1 + sum early-window impressions) | **Knowable at the decision moment because** only March 1–15 exposure is used, after the assumed reporting lag. |
| `ctr_pct` | 100 × early clicks / early impressions | **Knowable at the decision moment because** both counts are historical; 0.5 means 0.5%, not 50%. |
| `weighted_position` | sum(position × impressions) / sum(impressions), using valid position-days | **Knowable at the decision moment because** only historical rank observations and weights contribute. |
| `position_std` | population standard deviation of daily average positions on valid visible days | **Knowable at the decision moment because** volatility is computed entirely within March 1–15; a single measured day has std 0. |
| `active_day_fraction` | number of early days with impressions > 0 / 15 | **Knowable at the decision moment because** activity and complete tracking coverage are checked using the prior window only. |

The whole five-feature frame is built for eligible historical pages **before** looking at outcome eligibility. Later sufficient exposure is required only to score the proxy. An incomplete or absent outcome remains unknown. The displayed feature sample is five numeric aggregates with no IDs; the full frame and daily data are never exported.

In [5]:
# Feature/label construction SQL, separate from the three verification queries.
feature_sql = """
WITH available AS (
    SELECT * FROM march_source
    WHERE ga4_data_available IS TRUE
      AND gsc_impressions IS NOT NULL AND gsc_clicks IS NOT NULL
      AND gsc_impressions >= 0 AND gsc_clicks >= 0
), prior AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) AS prior_days,
           SUM(gsc_impressions) AS prior_impressions,
           SUM(gsc_clicks) AS prior_clicks,
           SUM(gsc_avg_position * gsc_impressions) FILTER
               (WHERE gsc_impressions > 0 AND gsc_avg_position > 0) /
           NULLIF(SUM(gsc_impressions) FILTER
               (WHERE gsc_impressions > 0 AND gsc_avg_position > 0), 0) AS weighted_position,
           STDDEV_POP(gsc_avg_position) FILTER
               (WHERE gsc_impressions > 0 AND gsc_avg_position > 0) AS position_std,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) / 15.0 AS active_day_fraction
    FROM available WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
), later AS (
    SELECT client_hash_id, content_hash_id, COUNT(*) AS outcome_days,
           SUM(gsc_impressions) AS future_impressions,
           SUM(gsc_clicks) AS future_clicks
    FROM available WHERE report_date BETWEEN DATE '2026-03-19' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT p.*, LN(1 + prior_impressions) AS log_impressions,
       100.0 * prior_clicks / prior_impressions AS ctr_pct,
       l.outcome_days, l.future_impressions, l.future_clicks
FROM prior p LEFT JOIN later l USING (client_hash_id, content_hash_id)
WHERE prior_days = 15 AND prior_impressions >= 500
  AND weighted_position BETWEEN 1 AND 20
ORDER BY client_hash_id, content_hash_id
"""
pages = con.sql(feature_sql).df()
assert len(pages) > 0, 'No pages meet the declared policy; review the data contract.'
assert not pages.duplicated(['client_hash_id', 'content_hash_id']).any()
assert pages['prior_days'].eq(15).all()
feature_frame = pages[FEATURE_COLS].copy()
assert len(FEATURE_COLS) == 5
assert np.isfinite(feature_frame.to_numpy()).all()
display(feature_frame.head(5).round(4))
display(pd.DataFrame({'feature': FEATURE_COLS,
                      'missing_values': feature_frame.isna().sum().to_numpy()}))
outcome_known = pages['outcome_days'].eq(13) & pages['future_impressions'].ge(100)
pages['future_ctr_pct'] = (100 * pages['future_clicks'] / pages['future_impressions']).where(outcome_known)
pages['future_low_ctr'] = pd.Series(pd.NA, index=pages.index, dtype='Int64')
pages.loc[outcome_known, 'future_low_ctr'] = pages.loc[outcome_known, 'future_ctr_pct'].lt(0.5).astype(int)
assert pages.loc[~outcome_known, 'future_low_ctr'].isna().all()
display(pd.DataFrame([{'historically_eligible_pages': len(pages),
                       'clients': pages['client_hash_id'].nunique(),
                       'known_outcome_pages': int(outcome_known.sum()),
                       'unknown_outcome_pages': int((~outcome_known).sum()),
                       'proxy_positive_pages': int(pages['future_low_ctr'].sum())}]))
print('Five features retained. Future measurements live only in the separate label/context frame.')

,log_impressions,ctr_pct,weighted_position,position_std,active_day_fraction
0,9.2819,2.7930,2.5229,0.6790,1.0
1,8.2548,0.7022,14.5404,5.1832,1.0
2,7.6816,3.4149,7.6022,1.9694,1.0
3,8.4530,0.9386,5.0595,1.4929,1.0
4,8.6933,1.1741,3.8187,2.2875,1.0


,feature,missing_values
0,log_impressions,0
1,ctr_pct,0
2,weighted_position,0
3,position_std,0
4,active_day_fraction,0


,historically_eligible_pages,clients,known_outcome_pages,unknown_outcome_pages,proxy_positive_pages
0,237,9,137,100,31


Five features retained. Future measurements live only in the separate label/context frame.


### The trap — add one label-derived column, then remove it

Use **leave-one-client-out validation**: each of the five measurable clients is held out once, so every labeled page receives a prediction from a model that never saw its client. This avoids a single grouped holdout containing only a few pages in an unbalanced panel. Pool out-of-client predictions for the quick score; large clients still dominate this page-level metric. Both models use identical folds, rows, depth and seed.

The honest tree sees only five prior-window features. The deliberate leak is exactly one column, `label_echo = 100 × future_low_ctr`; it reveals the later-window answer. A near-perfect result therefore demonstrates cheating. ROC AUC is primary; also show average precision, balanced accuracy and a train-prevalence baseline. This is one mid-panel development cycle, not a capstone test, editorial precision@20 or causal uplift.

In [6]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score

model_data = pages.loc[outcome_known].copy().reset_index(drop=True)
X = model_data[FEATURE_COLS].copy()
y = model_data['future_low_ctr'].astype(int)
groups = model_data['client_hash_id']
assert groups.nunique() >= 2 and y.nunique() == 2
splits = list(LeaveOneGroupOut().split(X, y, groups))
assert all(set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te])) for tr, te in splits)
assert all(y.iloc[tr].nunique() == 2 for tr, te in splits)
assert sorted(np.concatenate([te for tr, te in splits])) == list(range(len(X)))
display(pd.DataFrame([
    {'fold': i + 1, 'train_pages': len(tr), 'held_out_pages': len(te),
     'held_out_clients': groups.iloc[te].nunique(), 'held_out_positive_pages': int(y.iloc[te].sum())}
    for i, (tr, te) in enumerate(splits)
]))
print('Each client held out once; identical folds; tree seed 42; max_depth 4; min_samples_leaf 10.')

def out_of_client_probabilities(matrix):
    probabilities = np.full(len(y), np.nan)
    for tr, te in splits:
        model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
        model.fit(matrix.iloc[tr], y.iloc[tr])
        probabilities[te] = model.predict_proba(matrix.iloc[te])[:, 1]
    assert np.isfinite(probabilities).all()
    return probabilities

def evaluate(name, probabilities):
    return {'model': name,
            'roc_auc': roc_auc_score(y, probabilities),
            'average_precision': average_precision_score(y, probabilities),
            'balanced_accuracy': balanced_accuracy_score(y, probabilities >= 0.5)}

baseline_probabilities = np.full(len(y), np.nan)
for tr, te in splits:
    baseline_probabilities[te] = y.iloc[tr].mean()
baseline_result = evaluate('constant fold train-prevalence baseline', baseline_probabilities)
honest_probabilities = out_of_client_probabilities(X)
honest_result = evaluate('honest five-feature tree', honest_probabilities)

X_leaky = X.copy()
X_leaky['label_echo'] = 100 * y  # Intentionally unavailable at the decision moment.
leaky_result = evaluate('INVALID: five features + label_echo',
                        out_of_client_probabilities(X_leaky))
score_comparison = pd.DataFrame([baseline_result, honest_result, leaky_result])
display(score_comparison.round(4))
print(f"Leakage inflated out-of-client AUC by {leaky_result['roc_auc'] - honest_result['roc_auc']:.4f}.")
assert leaky_result['roc_auc'] >= 0.99
assert leaky_result['roc_auc'] > honest_result['roc_auc']


,fold,train_pages,held_out_pages,held_out_clients,held_out_positive_pages
0,1,60,77,1,13
1,2,135,2,1,1
2,3,82,55,1,17
3,4,135,2,1,0
4,5,136,1,1,0


Each client held out once; identical folds; tree seed 42; max_depth 4; min_samples_leaf 10.


,model,roc_auc,average_precision,balanced_accuracy
0,constant fold train-prevalence baseline,0.4028,0.2004,0.5000
1,honest five-feature tree,0.8112,0.5053,0.5618
2,INVALID: five features + label_echo,1.0000,1.0000,1.0000


Leakage inflated out-of-client AUC by 0.1888.


In [7]:
# Remove the trap and repeat the identical validation with only honest features.
X_leaky.drop(columns='label_echo', inplace=True)
assert list(X_leaky.columns) == FEATURE_COLS
restored_probabilities = out_of_client_probabilities(X_leaky)
del X_leaky
assert list(X.columns) == FEATURE_COLS
assert 'label_echo' not in feature_frame.columns
assert not {'future_ctr_pct', 'future_low_ctr', 'future_impressions',
            'future_clicks', 'client_hash_id', 'content_hash_id'}.intersection(X.columns)
assert np.allclose(restored_probabilities, honest_probabilities)
final_result = evaluate('honest after removing leak', restored_probabilities)
display(pd.DataFrame([final_result]).round(4))
# Retain an honest model fitted to this development cohort; score remains out-of-client.
final_model = DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42).fit(X, y)
display(Markdown(
    f"**Observed leakage lesson:** honest ROC AUC **{honest_result['roc_auc']:.4f}**, "
    f"invalid leaked AUC **{leaky_result['roc_auc']:.4f}**, and restored honest AUC "
    f"**{final_result['roc_auc']:.4f}** on the same leave-one-client-out predictions. "
    'The retained model uses only five historical features. '
    'These numbers evaluate the later low-CTR proxy, not actionable edits.'
))


,model,roc_auc,average_precision,balanced_accuracy
0,honest after removing leak,0.8112,0.5053,0.5618


**Observed leakage lesson:** honest ROC AUC **0.8112**, invalid leaked AUC **1.0000**, and restored honest AUC **0.8112** on the same leave-one-client-out predictions. The retained model uses only five historical features. These numbers evaluate the later low-CTR proxy, not actionable edits.

## 4. Data limits

**Named limitation — tracking-coverage selection:** this is an unbalanced panel. Requiring all 15 early days with GA4 available excludes GSC-only histories and clients whose analytics began mid-window. Requiring all 13 outcome days and at least 100 future impressions further selects the measurable evaluation cohort. That later filter can cause survivorship/exposure bias; it never defines the historical feature pool. The query outputs and coverage counts quantify what survives rather than treating unavailable rows as zero traffic.

**What the slice cannot establish:** one short March development cycle cannot demonstrate seasonal or future-month generalization. Average position hides query/device/intent mix, and positions missing on visible days can bias the weighted mean. The fixed low-CTR threshold is not position-adjusted opportunity, an independent editorial judgment, or evidence that a snippet edit would improve traffic. A future version should compare position-adjusted queues and impressions-only baselines using independent editorial precision@20, then evaluate on later unseen months and clients. The final June month remains unused.

**Timing limitation:** report dates and a release freshness cut do not prove when each historical observation was actually ingested. The three-day gap is a conservative assumption; a deployment needs real availability timestamps.

**Small-cohort limitation:** the declared policy leaves only 137 measurable pages across five clients. Leave-one-client-out validation uses them all, but individual folds may contain one class or only one or two pages; the pooled AUC is a noisy development diagnostic. Do not interpret a high score as reliable client/month generalization.

In [8]:
display(pd.DataFrame([
    {'stage': 'historical feature pool', 'pages': len(pages),
     'clients': pages['client_hash_id'].nunique()},
    {'stage': 'measurable future proxy cohort', 'pages': len(model_data),
     'clients': model_data['client_hash_id'].nunique()},
    {'stage': 'out-of-client predictions (all folds)', 'pages': len(y),
     'clients': groups.nunique()}
]))
print(f'Outcome coverage among historical candidates: {outcome_known.mean():.2%}.')
print('Independent editorial precision@20: unavailable — no review judgments supplied.')
print('Causal edit uplift: unmeasured. June final-month data: untouched.')


,stage,pages,clients
0,historical feature pool,237,9
1,measurable future proxy cohort,137,5
2,out-of-client predictions (all folds),137,5


Outcome coverage among historical candidates: 57.81%.
Independent editorial precision@20: unavailable — no review judgments supplied.
Causal edit uplift: unmeasured. June final-month data: untouched.


## 5. Self-check

The following code checks the computed requirements. A completed execution, saved with outputs, is the evidence; this notebook does not mark unexecuted checks as passed.

References for the implementation: [Hugging Face local authentication](https://huggingface.co/docs/huggingface_hub/en/guides/cli), [download only selected release files](https://huggingface.co/docs/huggingface_hub/en/guides/download), [leave-one-client-out validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html). Contract guidance: this repo's `skills/writing-data-contracts/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md`; assignment-card March/June guidance takes precedence over older sample-iteration hints.

In [9]:
checks = {
    'three verification outputs exist': all(len(t) > 0 for t in
        [grain_check, counts_check, availability_check]),
    'grain is unique and complete': grain_check.loc[0, 'duplicate_key_groups'] == 0
        and grain_check.loc[0, 'null_key_groups'] == 0,
    'availability SQL explicitly uses IS TRUE': 'ga4_data_available IS TRUE' in availability_sql,
    'exactly five finite historical features': X.shape[1] == 5 and np.isfinite(X.to_numpy()).all(),
    'feature and outcome windows disjoint': FEATURE_END < DECISION_DATE,
    'held-out clients disjoint': all(set(groups.iloc[tr]).isdisjoint(set(groups.iloc[te])) for tr, te in splits),
    'unknown labels remain missing': pages.loc[~outcome_known, 'future_low_ctr'].isna().all(),
    'leak demonstrated and removed': leaky_result['roc_auc'] >= 0.99 and 'X_leaky' not in globals(),
    'honest score restored': np.isclose(final_result['roc_auc'], honest_result['roc_auc']),
    'June/sample excluded': MONTH == '2026-03' and all('month=2026-03' in str(p) for p in daily_files),
}
display(pd.DataFrame({'check': list(checks), 'passed': list(checks.values())}))
assert all(checks.values())
print('All computational self-checks passed. Save executed outputs, commit only this notebook, then submit.')
con.close()

,check,passed
0,three verification outputs exist,True
1,grain is unique and complete,True
2,availability SQL explicitly uses IS TRUE,True
3,exactly five finite historical features,True
4,feature and outcome windows disjoint,True
5,held-out clients disjoint,True
6,unknown labels remain missing,True
7,leak demonstrated and removed,True
8,honest score restored,True
9,June/sample excluded,True


All computational self-checks passed. Save executed outputs, commit only this notebook, then submit.
